# Experiment B: select models on domain-disjoint data

Train and select the two classifiers using the frozen training and model-selection partitions on Kaggle CPU. Attach a verified checkpoint or regenerate the same split from the source data.


In [ ]:
STAGE = 'train-b'


In [ ]:
import base64, hashlib, io, json, os, sys, subprocess, zipfile
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
if Path('/kaggle/working').is_dir() and Path('/kaggle/input').is_dir():
    RUNTIME, BASE = 'kaggle', Path('/kaggle/working')
elif Path('/content').is_dir() and (os.environ.get('COLAB_RELEASE_TAG') or os.environ.get('COLAB_BACKEND_VERSION')):
    RUNTIME, BASE = 'colab', Path('/content')
else:
    raise RuntimeError('Kaggle/Colab execution context not verified; no installation or training started')
WORK = BASE / 'hufs-compsec-url-cascade'
WORK.mkdir(exist_ok=True)
RUN_ID = STAGE + '-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
print({'runtime':RUNTIME,'work':str(WORK),'stage':STAGE,'run_id':RUN_ID})


In [ ]:
# Verify the pinned code bundle and every file inside it.
PAYLOAD_SHA256 = '57d1d76fee99ccc5d5b415412ad704276f7ac52506544928a9a30c1306d53c99'
PAYLOAD = '...'
EXPECTED_FILES = {'configs/upstream.json': '18566709f55999e0565d9dcc3ded7f0b03e52b97d5ffd0bd2467a7272707fda5', 'requirements-gpu-cu118.in': 'b9e29b21f4ed14e0f13c4a2ec020fd77303488bf324ded55127ca87954aee250', 'requirements-linux.in': '9f4c6e84fc22a9ff895538b960dfead3023ac0c29edcc2c8622964f82a37fe18', 'scripts/audit_frozen.py': '8075802a0a84dc3ca418b85209e0a2a75c43c27093da579a197f60df585ee7bb', 'scripts/build_cloud_notebook.py': '95a6f537f2afa204fe19035e7fdc762cb13894238fe6f36d2fe952dd05eb50e6', 'scripts/check_restored_smoke.py': '86370da35e327ae7fd730c7fde3579efb405451b2b39673830313667effecda2', 'scripts/cloud_probe.py': 'b07fed2b8d0126f8019a9f4719b9a270e0b76f270053ff2373f7cf3b7e56a0a3', 'scripts/cloud_run.py': 'eb9ba57c70f72fd18dcab3a60e5105073ff0b5620367bcf6a101b6508962c313', 'scripts/fetch_assets.py': 'e25bfc54e9567af8b70d5900183b35be123c5a768c8b828c329e727985d47e8a', 'scripts/fetch_reference.py': '92cef6de421399adc0b9ec0a884b5ed4f5b5b58595fa6dc26a8cd88fd549e073', 'scripts/preflight.py': 'ee087db84d2856a433a4676164948e0428d28ed50ddb2f341fc76790382ba812', 'scripts/prepare_b.py': 'b838201f9a9c455eb4b85ff7500238988f32bc0f9f1cbf20a42b03942a4837ba', 'scripts/smoke_pilot.py': 'c235645f6dd680fc056d11bb3fe8441d0fda72f838c07a7bc68eab5b51a1d19a', 'scripts/submit_job.py': '1ea86638a58b0003ef4dbf98a6ce264a8a4f745924ffaa2eb743410bf3046ffb', 'scripts/train_a.py': '167a98d9d7caeda6e85bed53ee8c4f76cc468a0dcee76bee7fda4a4c5ac0329c', 'scripts/train_b.py': '130d7049d03af0a2fd7ef91d11874e4aad46e7dda2bdc82f1f79dc54eca9745f', 'scripts/verify_assets.py': 'bd15660683cd0ccfafd1ae21bf3c3c35a55bc7188b0899e87d9b571071a463d0', 'urlcascade/__init__.py': 'f43086e3ae07c4bbfe3443ba1dd4539372ca8a91edc20f3c503eb0a22d97d5b8', 'urlcascade/adapters.py': 'b0356dbc18177ef5f89de1aaeaafd7eadfd237f5d5ec4b4c6e81e71a7493c083', 'urlcascade/benchmark.py': '3171edfc295bd0c4dc4727375ef7ed38af4c1ec36ecac02e2ad2d4c1737b700d', 'urlcascade/core.py': 'ee06d57d4d59dfea86c68f24dc82544f160bb03ea271db4222c82cec2e74a590', 'urlcascade/data.py': '13d82e62472be176911b9372310ca22cec5b5deab36d44ac91d661de2a94dafd', 'urlcascade/features.py': 'ac6a878bb8f89888fbc2b3169211eec0cee6e01d71f1fd5f1ff46b63896498aa', 'urlcascade/runtime.py': '2dd7297ffceb189bea227fb2576533477b26f2ba79209afecda2bcb56667b74c'}
raw = base64.b64decode(PAYLOAD, validate=True)
assert hashlib.sha256(raw).hexdigest() == PAYLOAD_SHA256, 'payload hash mismatch'
CODE = WORK / ('code-' + PAYLOAD_SHA256[:16])
CODE.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as archive:
    assert set(archive.namelist()) == set(EXPECTED_FILES), 'unexpected payload members'
    for member in archive.infolist():
        path = PurePosixPath(member.filename)
        assert not path.is_absolute() and '..' not in path.parts, 'unsafe archive path'
        target = CODE / path
        assert target.resolve().is_relative_to(CODE.resolve()), 'unsafe output path'
        data = archive.read(member)
        assert hashlib.sha256(data).hexdigest() == EXPECTED_FILES[member.filename]
        if target.exists():
            assert hashlib.sha256(target.read_bytes()).hexdigest() == EXPECTED_FILES[member.filename], 'existing code mismatch'
        else:
            target.parent.mkdir(parents=True,exist_ok=True)
            target.write_bytes(data)
print({'code_files':len(EXPECTED_FILES),'code_sha256':PAYLOAD_SHA256})


In [ ]:
import hashlib
from datetime import datetime, timezone
from pathlib import Path
import shutil
import sys
import tempfile
import zipfile

assert RUNTIME == 'kaggle'
def file_digest(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for data in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(data)
    return digest.hexdigest()



In [ ]:
def run_stage(stage, *extra):
    run_id = stage + '-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    args = [sys.executable, str(CODE/'scripts/cloud_run.py'), '--runtime', RUNTIME,
            '--work', str(WORK), '--stage', stage, '--run-id', run_id, '--cpu-only', *extra]
    subprocess.run(args, check=True)
    return WORK/'runs'/run_id

setup = run_stage('setup')
candidates = list(Path('/kaggle/input').rglob('hufs-w1-w3-checkpoint.zip'))
if candidates:
    assert len(candidates) == 1, 'Expected exactly one attached checkpoint'
    assert file_digest(candidates[0]) == '4841dbf97f6ff084cc0b198a9b359f6a0c6d36d95332b07865b7c7e5d7c6109b'
    split = WORK / 'w2-frozen'
    split.mkdir(exist_ok=True)
    with zipfile.ZipFile(candidates[0]) as archive:
        manifests = [name for name in archive.namelist() if name.endswith('/split/manifest.json')]
        assert len(manifests) == 1, 'Expected one frozen split'
        prefix = manifests[0].rsplit('/', 1)[0] + '/'
        for name in ('manifest.json', 'train.jsonl', 'model_selection.jsonl'):
            with archive.open(prefix + name) as src, (split/name).open('wb') as dst:
                shutil.copyfileobj(src, dst)
    assert file_digest(split/'manifest.json') == '6440ae2948353ea42d5adb70a0e7a686d592496da94a8d24828707a0c6eef955'
    print('Restored the validated B partitions')
else:
    run_stage('fetch')
    split = run_stage('prepare-b') / 'split'
    run_stage('audit-b', '--split-dir', str(split))
trained = run_stage('train-b', '--split-dir', str(split))
print((trained/'experiment-b/models-frozen.json').read_text())


In [ ]:
target = WORK.parent/'hufs-b-frozen-models.zip'
with zipfile.ZipFile(target, 'x', zipfile.ZIP_DEFLATED) as archive:
    for folder in (CODE, setup, trained):
        for path in sorted(folder.rglob('*')):
            if path.is_file() and '__pycache__' not in path.parts and not path.is_relative_to(CODE/'upstream'):
                archive.write(path, path.relative_to(WORK))
with zipfile.ZipFile(target) as archive:
    assert archive.testzip() is None
print('Saved frozen B models', file_digest(target))
archive_staging = Path(tempfile.mkdtemp(prefix='hufs-completed-b-'))
shutil.move(str(WORK), str(archive_staging/'session-work'))
